In [1]:
import sys
sys.path.append('..')

import pickle
import numpy as np
from matplotlib.pyplot import cm
import pandas as pd
from numpy import linspace
import matplotlib.pyplot as plt

from utils.display_tools import load_best_forecasts,  display_predictions_2


In [235]:
# TODO SOMETHNG IS OFF HERE WITH THE SEARCH SPACE (Some combinations are missing. Find out what happens here and fix it for the revision.)

In [2]:
%load_ext autoreload
%autoreload 2

In [14]:
from os import listdir
from os.path import isfile, join
from utils.general_tools import results_to_pd


In [3]:
lot = pickle.load(open("../saves_and_results/cache_lot.p", "rb"))
data = pickle.load(open("../saves_and_results/cache_endog.p", "rb"))
exogs = pickle.load(open("../saves_and_results/cache_exogs.p", "rb"))

In [28]:
example_id = "10905"

In [32]:
methods = ["ada", "forest", "linear", "sarimax", "var"]

In [101]:
full_stack = []
for m in methods:
    res = []
    for x in ["", "_univariate","_remove_T","_remove_W"]:
        try:
            res.append(results_to_pd(pickle.load(open("results/journal/journal_new" + x + "/Moehne_desc1_" + m + "_results_stack.p", "rb"))))
        except:
            print("Error: " + m + x)
            continue
    res = pd.concat(res)
    res = res.loc[res["Index"] == example_id]
    res["method"] = m
    full_stack.append(res)

5184
32
1728
1728
3888
24
1296
1296
648
4
216
216
3888
24
1296
1296
1944
Error: var_univariate
648
648


In [102]:
full_stack = pd.concat(full_stack)[['P', 'D', 'Q', 'STAU', 'T',
       'M_Stau', 'M_T', 'Decompose', 'Interaction', 'Estimator', 'n_estimator',
       'max_depth', 'learning_rate', 'method']].reset_index(drop=True)

In [105]:
full_stack["STAU"] = full_stack["STAU"].astype(str)
full_stack["T"] =full_stack["T"].astype(str)

In [245]:
final = []
for method_name in full_stack["method"].unique():
    method = full_stack[full_stack["method"] == method_name]
    method = method[[x for x in method.columns if len(method[x].unique()) > 1]]
    method = pd.DataFrame([[list(method[x].unique())] for x in method.columns], index = method.columns, columns=[method_name])
    final.append(method)

In [263]:
search = pd.concat(final, axis=1)
search[search.isnull()] = "-"


In [264]:
search[search.isnull()] = "-"


In [285]:
search.loc["STAU"] = "[-,0,1,2]"
search.loc["T"] = "[-,0,1,2]"
search.loc["max_depth", "forest"] = ["-", 3, 7]
search.loc["learning_rate", "ada"] = [1, 0.1]
search = search[["linear", "sarimax", "var", "forest", "ada"]]

In [287]:
search 


,linear,sarimax,var,forest,ada
P,"[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]"
STAU,"[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]"
T,"[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]"
M_Stau,"[0, 7]","[0, 7]","[0, 7]","[0, 7]","[0, 7]"
M_T,"[0, 7]","[0, 7]","[0, 7]","[0, 7]","[0, 7]"
Decompose,"[0, 1]","[0, 1]","[0, 1]","[0, 1]","[0, 1]"
Interaction,"[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]"
Estimator,-,-,-,-,"[dt, linear]"
n_estimator,-,-,-,"[50, 250]","[50, 250]"
learning_rate,-,-,-,-,"[1, 0.1]"


In [289]:
search["Decomposin"]

,linear,sarimax,var,forest,ada
P,"[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]"
STAU,"[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]"
T,"[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]","[-,0,1,2]"
M_Stau,"[0, 7]","[0, 7]","[0, 7]","[0, 7]","[0, 7]"
M_T,"[0, 7]","[0, 7]","[0, 7]","[0, 7]","[0, 7]"
Decompose,"[0, 1]","[0, 1]","[0, 1]","[0, 1]","[0, 1]"
Interaction,"[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]","[0, 1, 2]"
Estimator,-,-,-,-,"[dt, linear]"
n_estimator,-,-,-,"[50, 250]","[50, 250]"
learning_rate,-,-,-,-,"[1, 0.1]"


In [290]:
print(search.to_latex())

\begin{tabular}{llllll}
\toprule
{} &     linear &    sarimax &        var &     forest &           ada \\
\midrule
P             &  [0, 1, 2] &  [0, 1, 2] &  [0, 1, 2] &  [0, 1, 2] &     [0, 1, 2] \\
STAU          &  [-,0,1,2] &  [-,0,1,2] &  [-,0,1,2] &  [-,0,1,2] &     [-,0,1,2] \\
T             &  [-,0,1,2] &  [-,0,1,2] &  [-,0,1,2] &  [-,0,1,2] &     [-,0,1,2] \\
M\_Stau        &     [0, 7] &     [0, 7] &     [0, 7] &     [0, 7] &        [0, 7] \\
M\_T           &     [0, 7] &     [0, 7] &     [0, 7] &     [0, 7] &        [0, 7] \\
Decompose     &     [0, 1] &     [0, 1] &     [0, 1] &     [0, 1] &        [0, 1] \\
Interaction   &  [0, 1, 2] &  [0, 1, 2] &  [0, 1, 2] &  [0, 1, 2] &     [0, 1, 2] \\
Estimator     &          - &          - &          - &          - &  [dt, linear] \\
n\_estimator   &          - &          - &          - &  [50, 250] &     [50, 250] \\
learning\_rate &          - &          - &          - &          - &      [1, 0.1] \\
max\_depth     &          - & 

/tmp/ipykernel_1898187/529449091.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(search.to_latex())


In [234]:
c = full_stack[full_stack["method"] == "linear"]
c = c[[x for x in c.columns if len(c[x].unique()) > 1]]
c.loc[(c["T"] == "[]") & (c["STAU"] == "[]")]

,P,STAU,T,M_Stau,M_T,Decompose,Interaction
15824,1,[],[],0,0,0,0
15825,1,[],[],0,0,1,0
15826,2,[],[],0,0,0,0
15827,2,[],[],0,0,1,0


In [230]:
c.loc[(c["T"] == "[]") & (c["STAU"] == "[]")]

,P,STAU,T,M_Stau,M_T,Decompose,Interaction
15824,1,[],[],0,0,0,0
15825,1,[],[],0,0,1,0
15826,2,[],[],0,0,0,0
15827,2,[],[],0,0,1,0


In [219]:
search["linear"].dropna().prod()

1152.0

In [173]:
3 * 4 * 4 *2 * 2 * 2 * 3 * 2 * 2 * 2

9216

In [91]:
stack = []
for col in check:
    stack.append(list(check[col].unique()))

,P,STAU,T,M_Stau,M_T,Decompose,Interaction,Estimator,n_estimator,learning_rate,method
0,0,[0],[0],0,0,0,0,dt,50,1.0,ada
1,0,[0],[0],0,0,0,0,dt,250,1.0,ada
2,0,[0],[0],0,0,0,0,dt,50,0.1,ada
3,0,[0],[0],0,0,0,0,dt,250,0.1,ada
4,0,[0],[0],0,0,0,0,linear,50,1.0,ada
...,...,...,...,...,...,...,...,...,...,...,...
25639,2,[],[0 1 2],7,7,0,1,None,None,None,var
25640,2,[],[0 1 2],7,7,0,2,None,None,None,var
25641,2,[],[0 1 2],7,7,1,0,None,None,None,var
25642,2,[],[0 1 2],7,7,1,1,None,None,None,var
